In [ ]:
import os
import sys
import platform

sys.path.append(os.path.dirname(os.getcwd()))
from benchmark_utils.sync_utils import plotFramesShiftToSyncrhonizeAllSubjectsOneActivity,getMainJointFromMotAndMainBonesFromCSV, getSamplesToSynchronize, SynchronizeAndCutSignals
from benchmark_utils.compare_utils import *
import benchmark_utils.file_utils as fileutil 
import benchmark_utils.plot_utils as plotutil
import benchmark_utils.signal_utils as signalutil
import benchmark_utils.evaluation_utils as evaluation_utils

# 1) Parámetros generales
dataset_path = '/Users/mario/Documents/vidimu_pipeline_COPIA/benchmark'

# 2) Nuevos directorios de entrada
in_path_bodytrack       = os.path.join(dataset_path, 'jointangles', 'jointangles_bodytrack')
in_path_motionbert      = os.path.join(dataset_path, 'jointangles', 'jointangles_motionbert')
in_path_mmpose          = os.path.join(dataset_path, 'jointangles', 'jointangles_mmpose')
in_path_motionagformer  = os.path.join(dataset_path, 'jointangles', 'jointangles_motionagformer')
in_path_imus            = os.path.join(dataset_path, 'jointangles', 'jointangles_imus')

# 3) Directorios de salida
out_path_eval = os.path.join(dataset_path, 'results', 'evaluation', 'PARAELTFG')
out_path_comp = os.path.join(dataset_path, 'results', 'comparison','PARAELTFG')
os.makedirs(out_path_eval, exist_ok=True)
os.makedirs(out_path_comp, exist_ok=True)

# 4) Definición de actividades y sujetos
lower_activities   = ["A01","A02","A03","A04"]
upper_activities   = ["A05","A06","A07","A08","A09","A10","A11","A12","A13"]
dataset_activities = lower_activities + upper_activities
activities_legend  = [
    "walk_forward", "walk_backward", "walk_along","sit_to_stand",
    "move_right_arm","move_left_arm","drink_right_arm","drink_left_arm",
    "assemble_both_arms","throw_both_arms",
    "reachup_right_arm","reachup_left_arm","tear_both_arms"
]
activities_dict    = dict(zip(dataset_activities, activities_legend))
selected_subjects  = [f"S{n}" for n in range(40,58) if n!=43 and n!=45]  # S40...S57 sin S43 y S45

# 5) Parámetros de sincronización/filtrado
RMSE_SAMPLES    = 180
FINAL_LENGTH    = 180
MAX_SYNC_OVERLAP= 15

# 6) Bucle principal: cálculo y guardado de métricas por actividad
per_activity_summaries = {}
for act, legend in zip(dataset_activities, activities_legend):
    print(f"Processing Activity {act}: {legend}")
    metrics_results = evaluation_utils.calculateAndPlotAllMetrics(
        csv_bodytrack_path=in_path_bodytrack,
        csv_motionbert_path=in_path_motionbert,
        csv_mmpose_path=in_path_mmpose,
        csv_motionagformer_path=in_path_motionagformer,
        imu_inpath=in_path_imus,
        subjects=selected_subjects,
        activity=act,
        activity_legend=legend,
        RMSE_SAMPLES=RMSE_SAMPLES,
        MAX_SYNC_OVERLAP=MAX_SYNC_OVERLAP,
        FINAL_LENGTH=FINAL_LENGTH,
        out_path=out_path_eval,
        filename_prefix=f"{act}_BarCharts"
    )
    # Resumen y tabla
    summary_df = evaluation_utils.createSummaryTable(metrics_results)
    evaluation_utils.plotSummaryTable(
        summary_df,
        title=f"Aggregated Performance Metrics for Activity {act}: {legend}",
        out_path=out_path_eval,
        filename_prefix=f"{act}_SummaryTable"
    )
    per_activity_summaries[act] = summary_df


In [ ]:

# 7) Generar tablas comparativas globales (RMSE, MAE y R²)

mae_table = evaluation_utils.createActivityMAEComparisonTable(
    per_activity_summaries, activities_dict,
    out_path=out_path_comp,
    filename_prefix="Overall_PerActivityMAE_Table"
)

r2_table = evaluation_utils.createActivityR2ComparisonTable(
    per_activity_summaries, activities_dict,
    out_path=out_path_comp,
    filename_prefix="Overall_PerActivityR2_Table"
)

print("Tablas comparativas generadas en:", out_path_comp)
